# Day2 모델링 (6) - 제출 3·4 재현과 리더보드 해석

06에서 만든 제출 2 이후에 낸 **제출 3, 제출 4를 재현**하고, 리더보드 결과를 어떻게 읽어야 하는지 정리한다. 새로운 모델 탐색은 하지 않는다.

- 제출 3: Tanimoto SVM + ExtraTrees + RandomForest의 확률 평균(GBM 없음), 임계값은 OOF에서 정한다.
- 제출 4: 제출 2 구성(LightGBM + XGBoost + SVM)과 제출 3 구성의 확률을 반반 평균, 임계값은 OOF에서 정한다.
- 모든 임계값은 OOF 예측만으로 정했다. 리더보드 점수로 임계값이나 가중치를 맞추지 않았다. 다만 제출 4의 구성(제출 2·3 평균)은 제출 3의 결과를 본 뒤에 정했다.
- 3~5절의 분석은 리더보드에서 역산한 값(테스트 양성 개수 N)을 쓴다. **분석과 해석용이고 모델 학습에는 들어가지 않는다.**
- 이 노트북은 05, 06을 먼저 실행해서 `data/cache/`의 `oof_03.pkl`, `oof_05.pkl`, `final_proba_pred.pkl`이 있어야 한다.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_all  # noqa: E402
from models import TanimotoSVC  # noqa: E402
from preprocess import build_pipeline  # noqa: E402
from validate_submission import validate  # noqa: E402

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
SEED = 42
TAB = ROOT / "reports/tables"
CACHE = ROOT / "data/cache"
d = load_all(ROOT, seed=SEED, extra=True)
y = d.y_tr.values
best = json.loads((TAB / "best_params.json").read_text(encoding="utf-8"))

# 03(기본 6모델)과 05(튜닝 2모델, SVM)의 OOF 확률. 같은 학습 80%와 같은 5-Fold 분할이라 행 순서가 같다.
oof = pd.read_pickle(CACHE / "oof_03.pkl")
oof.update(pd.read_pickle(CACHE / "oof_05.pkl"))
L, X, S = "LightGBM(tuned)", "XGBoost(tuned)", "TanimotoSVC"

THS = np.round(np.arange(0.30, 0.61, 0.01), 2)


def pick_threshold(p):
    # 06과 같은 규칙: OOF F1 곡선의 5점 이동평균 정점(평탄 구간의 중앙). 홀드아웃과 리더보드는 쓰지 않는다.
    f1s = np.array([f1_score(y, (p >= t).astype(int)) for t in THS])
    i = np.convolve(f1s, np.ones(5) / 5, mode="same").argmax()
    return float(THS[i]), float(f1s[i]), float(f1s.max())


# 제출 2·3·4 구성의 OOF 확률
oof_g2 = np.mean([oof[L], oof[X], oof[S]], axis=0)
oof_g3 = np.mean([oof[S], oof["ExtraTrees"], oof["RandomForest"]], axis=0)
oof_g4 = (oof_g2 + oof_g3) / 2
THR2, THR3, THR4 = pick_threshold(oof_g2)[0], pick_threshold(oof_g3)[0], pick_threshold(oof_g4)[0]
print("OOF에서 정한 임계값  제출2:", THR2, "제출3:", THR3, "제출4:", THR4)

OOF에서 정한 임계값  제출2: 0.43 제출3: 0.44 제출4: 0.45


## 1. 제출 3·4 파일 만들기

전체 train(8,349건)으로 SVM, ExtraTrees, RandomForest를 한 번씩 학습한다. LightGBM과 XGBoost는 06에서 만든 제출 2 구성의 확률(`final_proba_pred.pkl`)을 그대로 쓴다. 파일이 이미 있으면 덮어쓰지 않고 새 계산 결과와 같은지만 비교한다.

In [2]:
SVC_GROUPS = ["fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]
BASE = ["cont", "rdkit", "fp"]
svc = build_pipeline(TanimotoSVC(C=best["svc_C"]), d.groups, SVC_GROUPS).fit(d.X, d.y)
et = build_pipeline(ExtraTreesClassifier(n_estimators=500, n_jobs=4, random_state=SEED), d.groups, BASE).fit(d.X, d.y)
rf = build_pipeline(RandomForestClassifier(n_estimators=500, n_jobs=4, random_state=SEED), d.groups, BASE).fit(d.X, d.y)
p_svc, p_et, p_rf = [m.predict_proba(d.X_pred)[:, 1] for m in (svc, et, rf)]
p_g2 = pd.read_pickle(CACHE / "final_proba_pred.pkl")  # 06의 제출 2 구성(LightGBM + XGBoost + SVM) 확률
p_g3 = (p_svc + p_et + p_rf) / 3
p_g4 = (p_g2 + p_g3) / 2


def save_or_compare(sub, name):
    path = ROOT / "submissions" / name
    if path.exists():
        old = pd.read_csv(path)
        diff = int((old["output"].values != sub["output"].values).sum())
        print(f"{name}: 이미 있음, 새 계산과 {'일치' if diff == 0 else f'{diff}행 다름'}")
    else:
        path.parent.mkdir(exist_ok=True)
        sub.to_csv(path, index=False)
        print(f"{name}: 저장")
    errors = validate(path)
    print("  검증", "통과" if not errors else errors, "| 양성", int(sub["output"].sum()), f"({sub['output'].mean():.3f})")


sub2_pred = (p_g2 >= THR2).astype(int)
for n, p, t in [(3, p_g3, THR3), (4, p_g4, THR4)]:
    sub = pd.DataFrame({"SMILES": d.smiles_pred, "output": (p >= t).astype(int)})
    save_or_compare(sub, f"predict_output_정태호_{n}.csv")
    print(f"  제출 2 구성과 다른 행: {int((sub['output'].values != sub2_pred).sum())}")

predict_output_정태호_3.csv: 이미 있음, 새 계산과 일치
  검증 통과 | 양성 544 (0.587)
  제출 2 구성과 다른 행: 38
predict_output_정태호_4.csv: 이미 있음, 새 계산과 일치
  검증 통과 | 양성 544 (0.587)
  제출 2 구성과 다른 행: 22


## 2. 리더보드 점수에서 혼동행렬 역산

리더보드는 F1, Accuracy, Precision, Recall만 보여 준다. 테스트 양성 개수를 N이라 하면 TP, FP가 정수로 정해지므로, **4회 제출의 네 지표를 모두 소수 넷째 자리까지 다시 만들어 내는 N**을 찾는다. 이 N은 분석(3~5절)에만 쓴다.

In [3]:
N_ALL = len(d.X_pred)
LB = pd.DataFrame([
    {"제출": 1, "구성": "HistGB(수치+RDKit+핑거프린트), 임계값 0.5", "F1": 0.8284, "Accuracy": 0.8123, "Precision": 0.8061, "Recall": 0.8519},
    {"제출": 2, "구성": "LightGBM+XGBoost+SVM 평균(튜닝)", "F1": 0.8287, "Accuracy": 0.8069, "Precision": 0.7844, "Recall": 0.8783},
    {"제출": 3, "구성": "SVM+ExtraTrees+RandomForest 평균", "F1": 0.8351, "Accuracy": 0.8155, "Precision": 0.7960, "Recall": 0.8783},
    {"제출": 4, "구성": "제출 2 구성과 제출 3 구성 평균", "F1": 0.8312, "Accuracy": 0.8112, "Precision": 0.7923, "Recall": 0.8742},
])


def confusion(row, n_pos):
    tp = round(row["Recall"] * n_pos)
    pred_pos = round(tp / row["Precision"])
    fp = pred_pos - tp
    fn = n_pos - tp
    tn = N_ALL - n_pos - fp
    return tp, fp, fn, tn


def matches(row, n_pos):
    tp, fp, fn, tn = confusion(row, n_pos)
    if min(tp, fp, fn, tn) < 0:
        return False
    prec, rec = tp / (tp + fp), tp / (tp + fn)
    f1, acc = 2 * prec * rec / (prec + rec), (tp + tn) / N_ALL
    return all(abs(a - b) < 6e-5 for a, b in [(prec, row["Precision"]), (rec, row["Recall"]), (f1, row["F1"]), (acc, row["Accuracy"])])


cands = [n for n in range(300, 700) if all(matches(r, n) for _, r in LB.iterrows())]
print("네 제출의 지표를 모두 재현하는 테스트 양성 개수 N:", cands)
N_POS = cands[0]
rows = []
for _, r in LB.iterrows():
    tp, fp, fn, tn = confusion(r, N_POS)
    rows.append({**r.to_dict(), "양성 예측": tp + fp, "TP": tp, "FP": fp, "FN": fn, "TN": tn})
lb = pd.DataFrame(rows)
lb["OOF F1"] = [None, pick_threshold(oof_g2)[1], pick_threshold(oof_g3)[1], pick_threshold(oof_g4)[1]]
lb["임계값"] = [0.5, THR2, THR3, THR4]
lb.round(4).to_csv(TAB / "submission_leaderboard.csv", index=False)
print(f"테스트 양성 {N_POS}건({N_POS / N_ALL:.3f}), 음성 {N_ALL - N_POS}건. 학습 데이터의 양성 비율은 {d.y.mean():.3f}")
lb[["제출", "임계값", "양성 예측", "TP", "FP", "FN", "TN", "F1", "Accuracy", "Precision", "Recall", "OOF F1"]].round(4)

네 제출의 지표를 모두 재현하는 테스트 양성 개수 N: [493]
테스트 양성 493건(0.532), 음성 434건. 학습 데이터의 양성 비율은 0.544


,제출,임계값,양성 예측,TP,FP,FN,TN,F1,Accuracy,Precision,Recall,OOF F1
0,1,0.50,521,420,101,73,333,0.8284,0.8123,0.8061,0.8519,NaN
1,2,0.43,552,433,119,60,315,0.8287,0.8069,0.7844,0.8783,0.8328
2,3,0.44,544,433,111,60,323,0.8351,0.8155,0.7960,0.8783,0.8339
3,4,0.45,544,431,113,62,321,0.8312,0.8112,0.7923,0.8742,0.8357


**읽는 법**: 제출 2와 3은 맞힌 양성(TP)이 같고, 오답 양성(FP)만 8건 차이가 난다. 제출 3과 4는 18행이 달라서 TP 2건과 FP 2건이 바뀌었을 뿐이다. 점수 차이는 몇 건의 예측 차이에서 나온다.

테스트 표준오차를 보면 이 차이가 얼마나 작은지 알 수 있다. 927건에서 F1의 표준편차는 약 0.0126이고, 네 제출의 점수 폭(0.8284~0.8351)은 0.0067이다.

In [4]:
# 같은 확률 모델을 다른 927건 시험지로 보면 F1이 얼마나 흔들리는지(부트스트랩, 학습 데이터 OOF 사용)
rng = np.random.default_rng(SEED)
pred_oof = (oof_g2 >= THR2).astype(int)
boot = []
for _ in range(1000):
    idx = rng.choice(len(y), size=N_ALL, replace=True)
    boot.append(f1_score(y[idx], pred_oof[idx]))
print(f"OOF 예측을 927건씩 다시 뽑았을 때 F1 평균 {np.mean(boot):.4f}, 표준편차 {np.std(boot):.4f}, 95% 구간 [{np.percentile(boot, 2.5):.4f}, {np.percentile(boot, 97.5):.4f}]")
print("리더보드 네 제출의 F1 폭:", round(lb['F1'].max() - lb['F1'].min(), 4))

OOF 예측을 927건씩 다시 뽑았을 때 F1 평균 0.8332, 표준편차 0.0127, 95% 구간 [0.8071, 0.8577]
리더보드 네 제출의 F1 폭: 0.0067


## 3. 앙상블 구성 후보 비교 (OOF)

제출 2 이후에 시험한 앙상블 구성들이다. 모두 05, 03의 OOF 확률을 평균해서 만들고, 임계값은 구성마다 같은 규칙으로 정했다. "제출 2와 다른 행"은 OOF 6,679행에서 예측이 달라진 개수다.

In [5]:
ref_pred = (oof_g2 >= THR2).astype(int)
ET, RF, CB, HG = "ExtraTrees", "RandomForest", "CatBoost", "HistGB"
CAND = {
    "제출 2: LGBM+XGB+SVM": oof_g2,
    "+ ExtraTrees": np.mean([oof[k] for k in (L, X, S, ET)], axis=0),
    "+ ExtraTrees + RandomForest": np.mean([oof[k] for k in (L, X, S, ET, RF)], axis=0),
    "7모델 평균": np.mean([oof[k] for k in (L, X, S, ET, RF, CB, HG)], axis=0),
    "SVM 가중치 0.5": np.average([oof[L], oof[X], oof[S]], axis=0, weights=[0.25, 0.25, 0.5]),
    "SVM 0.4 + ET 0.2 + LGBM/XGB 0.2씩": np.average([oof[k] for k in (L, X, S, ET)], axis=0, weights=[0.2, 0.2, 0.4, 0.2]),
    "제출 3: SVM+ET+RF (GBM 없음)": oof_g3,
    "제출 4: 제출 2·3 구성 평균": oof_g4,
}
rows = []
for name, p in CAND.items():
    t, f1_t, f1_max = pick_threshold(p)
    pred = (p >= t).astype(int)
    rows.append({"구성": name, "AUC": roc_auc_score(y, p), "임계값": t, "F1@임계값": f1_t, "최대 F1": f1_max,
                 "양성 비율": pred.mean(), "제출 2와 다른 행": int((pred != ref_pred).sum())})
cand_df = pd.DataFrame(rows)
cand_df.round(4).to_csv(TAB / "submission_candidates.csv", index=False)
cand_df.round(4)

,구성,AUC,임계값,F1@임계값,최대 F1,양성 비율,제출 2와 다른 행
0,제출 2: LGBM+XGB+SVM,0.8889,0.43,0.8328,0.8334,0.5984,0
1,+ ExtraTrees,0.8884,0.42,0.8356,0.8356,0.6079,111
2,+ ExtraTrees + RandomForest,0.8879,0.45,0.8353,0.8356,0.5978,124
3,7모델 평균,0.8881,0.44,0.8340,0.8348,0.6053,148
4,SVM 가중치 0.5,0.8877,0.45,0.8349,0.8349,0.5890,123
5,SVM 0.4 + ET 0.2 + LGBM/XGB 0.2씩,0.8881,0.44,0.8354,0.8364,0.5956,131
6,제출 3: SVM+ET+RF (GBM 없음),0.8829,0.44,0.8339,0.8339,0.6113,340
7,제출 4: 제출 2·3 구성 평균,0.8882,0.45,0.8357,0.8357,0.5941,135


**읽는 법**: 어떤 구성이든 OOF F1은 0.833~0.836 안에서 움직이고, AUC는 0.883~0.889다. OOF 6,679행의 표준오차(약 0.005, 시험지 927건의 0.0126에서 환산)와 비슷한 크기라서 "이 구성이 더 낫다"고 말할 수 없다. 이 후보들은 OOF를 보고 골랐으므로 OOF F1이 조금 낙관적이다(선택 편향).

## 4. 양성 예측 개수(임계값)를 바꾸면 F1이 오를까

OOF 확률을 실제 양성 확률로 보정(Isotonic)하고, 테스트 양성이 N건이라는 가정으로 **양성을 k건 예측했을 때의 기대 F1**을 계산한다. 보정과 계산에는 라벨 없는 테스트 확률과 N만 쓴다.

In [6]:
iso = IsotonicRegression(out_of_bounds="clip").fit(oof_g2, y)
q = iso.predict(p_g2)
q = q * N_POS / q.sum()  # 보정 확률의 합을 테스트 양성 개수에 맞춘다
order = np.sort(q)[::-1]
cum = np.cumsum(order)
rows = [{"양성 예측 k": k, "기대 TP": cum[k - 1], "기대 F1": 2 * cum[k - 1] / (k + N_POS), "해당 임계값(제출 2 확률)": np.sort(p_g2)[::-1][k - 1]}
        for k in range(480, 621, 10)]
exp_df = pd.DataFrame(rows)
exp_df.round(4).to_csv(TAB / "submission_expected_f1.csv", index=False)
print("제출 2의 양성 예측:", int(sub2_pred.sum()), "| 제출 1의 양성 예측:", int(lb.loc[0, "양성 예측"]))
exp_df.round(4)

제출 2의 양성 예측: 552 | 제출 1의 양성 예측: 521


,양성 예측 k,기대 TP,기대 F1,해당 임계값(제출 2 확률)
0,480,399.0549,0.8203,0.6080
1,490,405.1374,0.8243,0.5913
2,500,411.0583,0.8279,0.5637
3,510,416.5711,0.8307,0.5337
4,520,421.4511,0.8321,0.5095
5,530,425.9763,0.8328,0.4782
6,540,430.2889,0.8331,0.4573
7,550,434.5965,0.8334,0.4417
8,560,438.6885,0.8332,0.4024
9,570,442.7706,0.8331,0.3722


**읽는 법**: 기대 F1은 530~580건 구간에서 0.833 안팎으로 평평하고 최대는 약 550건이다. 제출 2(552건)가 이미 그 자리이고, 521건을 예측한 제출 1과 552건을 예측한 제출 2의 점수(0.8284, 0.8287)가 같은 것과도 맞는다. **임계값을 옮겨서 더 얻을 수 있는 F1은 거의 없다.**

## 5. 구조가 비슷한 분자와 같은 분자: 이웃 정답을 쓰면 좋아질까

두 가지를 확인한다.

1. **유사도 구간별 정확도**: 가장 비슷한 학습 분자(ECFP Tanimoto)와의 유사도 구간에서 모델과 "이웃 정답 그대로 쓰기"의 정확도를 비교한다. 같은 구간의 테스트 개수로 가중하면 `predict_input`의 분포에 맞춘 점수가 된다(라벨 없는 입력 구조만 사용).
2. **같은 분자**: 입체 정보와 염을 제거한 정규화 SMILES가 학습 데이터와 같은 테스트 분자가 있는지, 있으면 학습 라벨과 모델 예측이 같은지 본다.

In [7]:
ecfp = [c for c in d.groups["fp"] if c.startswith("ecfp")]
A = d.X_tr[ecfp].values.astype(np.float32)
cnt = A.sum(1)
sim_nn = np.zeros(len(y))
lab_nn = np.zeros(len(y))
for tr, va in StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(A, y):  # OOF와 같은 분할
    inter = A[va] @ A[tr].T
    S_mat = inter / (cnt[va][:, None] + cnt[tr][None, :] - inter + 1e-9)
    j = S_mat.argmax(1)
    sim_nn[va] = S_mat.max(1)
    lab_nn[va] = y[tr][j]

A_all = d.X[ecfp].values.astype(np.float32)
P = d.X_pred[ecfp].values.astype(np.float32)
inter = P @ A_all.T
sim_test = (inter / (P.sum(1)[:, None] + A_all.sum(1)[None, :] - inter + 1e-9)).max(1)

edges = [0, 0.6, 0.7, 0.8, 0.9, 1.01]
b_oof, b_test = np.digitize(sim_nn, edges) - 1, np.digitize(sim_test, edges) - 1
pred2, pred3, pred4 = (oof_g2 >= THR2), (oof_g3 >= THR3), (oof_g4 >= THR4)
rows = []
for b in range(len(edges) - 1):
    m = b_oof == b
    rows.append({"최근접 유사도": f"{edges[b]:.1f}~{min(edges[b + 1], 1.0):.1f}", "OOF 행": int(m.sum()), "테스트 행": int((b_test == b).sum()),
                 "제출 2 구성 정확도": (pred2[m] == y[m]).mean(), "제출 3 구성 정확도": (pred3[m] == y[m]).mean(),
                 "이웃 정답 정확도": (lab_nn[m] == y[m]).mean()})
sim_df = pd.DataFrame(rows)
sim_df.round(4).to_csv(TAB / "similarity_bins.csv", index=False)
print("테스트 중 유사도 0.8 이상:", round((sim_test >= 0.8).mean(), 3), "| OOF 중 0.8 이상:", round((sim_nn >= 0.8).mean(), 3))
sim_df.round(3)

테스트 중 유사도 0.8 이상: 0.529 | OOF 중 0.8 이상: 0.409


,최근접 유사도,OOF 행,테스트 행,제출 2 구성 정확도,제출 3 구성 정확도,이웃 정답 정확도
0,0.0~0.6,1039,104,0.765,0.764,0.660
1,0.6~0.7,896,85,0.792,0.790,0.733
2,0.7~0.8,2010,248,0.790,0.794,0.736
3,0.8~0.9,2063,337,0.841,0.835,0.801
4,0.9~1.0,671,153,0.857,0.860,0.833


In [8]:
# 테스트 분포 가중 F1: 각 OOF 행에 (테스트 구간 비율 / OOF 구간 비율)을 곱한다. 임계값은 위에서 정한 값을 그대로 쓴다.
w = np.array([(b_test == b).mean() / (b_oof == b).mean() for b in range(len(edges) - 1)])[b_oof]


def weighted_f1(pred):
    tp, fp, fn = (w * (pred & (y == 1))).sum(), (w * (pred & (y == 0))).sum(), (w * (~pred & (y == 1))).sum()
    return 2 * tp / (2 * tp + fp + fn)


print("일반 OOF F1     제출2 %.4f | 제출3 %.4f | 제출4 %.4f" % tuple(f1_score(y, p) for p in (pred2, pred3, pred4)))
print("테스트분포 가중 제출2 %.4f | 제출3 %.4f | 제출4 %.4f" % tuple(weighted_f1(p) for p in (pred2, pred3, pred4)))

일반 OOF F1     제출2 0.8328 | 제출3 0.8339 | 제출4 0.8357
테스트분포 가중 제출2 0.8392 | 제출3 0.8398 | 제출4 0.8423


In [9]:
# 같은 분자 찾기: 최대 조각만 남기고(FragmentParent) 입체 정보를 포함/제외한 두 가지 키로 비교
tr_all = pd.read_csv(ROOT / "data/train.csv", usecols=["SMILES", "label"])
te_all = pd.read_csv(ROOT / "data/predict_input.csv", usecols=["SMILES"])


def keys(smiles, stereo):
    return [Chem.MolToSmiles(rdMolStandardize.FragmentParent(Chem.MolFromSmiles(s)), isomericSmiles=stereo) for s in smiles]


for stereo in (True, False):
    tr_key, te_key = keys(tr_all["SMILES"], stereo), keys(te_all["SMILES"], stereo)
    grp = tr_all.assign(k=tr_key).groupby("k")["label"].agg(["mean", "size"])
    hit = pd.Series(te_key).isin(grp.index).values
    dup = grp[grp["size"] > 1]
    mixed = int(((dup["mean"] > 0) & (dup["mean"] < 1)).sum())
    msg = f"입체 {'포함' if stereo else '제외'}: 테스트 중 학습과 같은 분자 {int(hit.sum())}건 | 학습 내 중복 키 {len(dup)}개 중 라벨이 갈리는 것 {mixed}개"
    if hit.any():
        train_label = pd.Series(te_key)[hit].map(grp["mean"]).values >= 0.5
        msg += f" | 제출 2 예측이 학습 라벨과 다른 행 {int(((p_g2[hit] >= THR2) != train_label).sum())}"
    print(msg)

입체 포함: 테스트 중 학습과 같은 분자 5건 | 학습 내 중복 키 23개 중 라벨이 갈리는 것 1개 | 제출 2 예측이 학습 라벨과 다른 행 0


입체 제외: 테스트 중 학습과 같은 분자 85건 | 학습 내 중복 키 259개 중 라벨이 갈리는 것 37개 | 제출 2 예측이 학습 라벨과 다른 행 0


## 6. 정리

- 제출 4회의 리더보드 F1은 0.8284 → 0.8287 → 0.8351 → 0.8312다. 폭 0.0067은 927건 시험지의 우연(표준편차 약 0.0126)보다 작아서 **모델 구성의 차이가 구별되지 않는다.** OOF 순서도 리더보드와 맞지 않았다.
- 임계값(양성 예측 개수)은 이미 평탄한 구간의 중앙에 있고, 이웃 정답이나 같은 분자의 라벨을 덮어쓰는 방법은 모든 유사도 구간에서 모델보다 정확도가 낮거나 이미 같은 답이었다.
- 더 오르려면 정보가 달라져야 한다(새로운 종류의 입력). 같은 분자를 다른 각도로 찍은 사진(기술자, 지문)을 섞는 방법은 AUC 0.886~0.889 근처에서 한계가 보인다.
- 선택 기준은 앞으로도 교차검증과 홀드아웃이다. 리더보드는 같은 분포의 시험지 한 장으로 얻은 참고값이다.